# T-Test

In this section we will be going through the process of an entire one sample t-test, plotting power curves, and calculating p-values using data.

A one sample t-test is a test used to determine if an unknown population mean is not equal to a value. In this test we reject or fail to reject a null hypothesis, which claims that the unknown population mean is equal to a certain value. A t-test uses the student t-distribution with the formula

$$
t=\frac{\bar{x}-\mu}{s/\sqrt{n}}
$$
<!-- 
We will first simulate conducting a t-test using the Python module [statsmodels](https://www.statsmodels.org/dev/index.html). In this simulated case we will have the following hypotheses:

$$
\begin{aligned}
H_0&: \mu=10 \\
H_1&: \mu\neq 10 \\
\end{aligned}
$$

The significance value we will be using is $0.05$. This means if our p-value is below this value, we reject $H_0$. Otherwise we fail to reject $H_0$. Run the code below a few times to simulate testing on different samples. Observe the variation in the results. -->

We will be using the Michelson speed of light dataset for our test. This dataset comprises 100 measurements from 5 trials conducted by Albert Michelson where each measurement values is given as (observed value - $299,000$) km/sec. Run the code below to extract our data. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.weightstats import DescrStatsW
import pandas as pd

In [ ]:
data = pd.read_csv("data/morley.csv")

print(data.head())

The true speed of light is $299,792$ km/sec, meaning our null hypothesis is $\mu=792$. Our hypotheses are as follows:

$$
\begin{aligned}
H_0 &: \mu =  792\\
H_1 &: \mu \neq 792\\
\end{aligned}
$$

Lets begin by plotting power curves that represent this scenario. Run the code block below to draw power curves for varying significance levels and calculate the effect sizes at which the power is $80\%$ for each significance level.

In [ ]:
sample = data["Speed"].sample(20, random_state=1).values
n = len(sample)
alphas = [0.05, 0.025, 0.01, 0.005]
df = n - 1
s = np.std(sample, ddof=1)
effect_sizes = []
mu0 = 792

plt.figure()

for alpha in alphas:
    mus = np.linspace(mu0 - 1.5 * s, mu0 + 1.5 * s, 200)
    t_crit = stats.t.ppf(1 - alpha / 2, df)
    ncp = (mus - mu0) / (s / np.sqrt(n))
    power_exact = 1 - stats.nct.cdf(t_crit, df, ncp) + stats.nct.cdf(-t_crit, df, ncp)
    plt.plot(mus, power_exact, label=f"Alpha = {alpha}")

    indices = np.argwhere(power_exact >= 0.8)
    effect_sizes = (mus[indices] - mu0) / s
    min_effect_size = np.min(np.abs(effect_sizes))
    print(
        f"Effect sizes for power >= 0.8 alpha={alpha}: effect size >= {min_effect_size:.2f}"
    )

plt.legend()

plt.ylabel("Power")
plt.title("Power Curves: Exact (One-Sample t-Test)")
plt.grid(True, alpha=0.3)
plt.show()

Now we can go about performing the t-test by calculating our test statistic and p-value. We will be performing a two-sided test. 

Once we have done so, we can form a conclusion on whether we reject or fail to reject our null hypothesis. Run the code block below to conduct this test. Run the code a few times and observe any variation in the results.

In [ ]:
sample = data["Speed"].sample(n=60)
mu0 = 792
print(f"mu1 = {np.mean(sample)}")
# perform t-test
desc_stats = DescrStatsW(sample)
t_stat, p_val, dof = desc_stats.ttest_mean(value=mu0, alternative="two-sided")

# result analysis
print("t-statistic = " + str(t_stat))
print("p-value = " + str(p_val))
if p_val < 0.05:
    print("Reject the null hypothesis")
else:
    print("Fail to reject the null hypothesis")